# <center >项目案例：基于LangChain构建NL2SQL数据分析系统</center>

<div align=center><img src="https://muyu20241105.oss-cn-beijing.aliyuncs.com/images/202511111509333.png" width=70%></div>

# 一、NL2SQL 项目架构

&emsp;&emsp;SQL Agent 数据分析系统是一个基于 **大语言模型（LLM）** 和 **RAG（检索增强生成）** 技术的智能数据分析平台。系统的核心价值在于：**将自然语言问题自动转换为准确的 SQL 查询，并对查询结果进行业务解读和可视化呈现**。用户无需掌握 SQL 语法，只需用日常语言提问（如："2024年销售额最高的10个产品是什么？"），系统即可自动完成数据查询、分析和报告生成。

&emsp;&emsp;与传统的 BI 工具相比，本系统具备三大优势：**自然交互**（无需学习复杂的拖拽式界面）、**智能推理**（基于 Agent 的多步推理能力）、**持续学习**（通过 RAG 技术不断积累领域知识）。

&emsp;&emsp;系统采用 **自底向上** 的四层架构设计，每一层职责明确、边界清晰，确保系统的可扩展性和可维护性。

<div align=center><img src="https://muyu20241105.oss-cn-beijing.aliyuncs.com/images/202511111536372.png" width=70%></div>

&emsp;&emsp;用户通过 **自然语言输入框** 提出数据分析需求，系统实时展示 Agent 的思考过程（如"正在检索表结构"、"正在生成 SQL"），最终以 **数据表格、可视化图表、自然语言解释** 三种形式呈现查询结果。此外，用户还可以通过 **训练数据管理面板** 手动添加 DDL 定义、业务文档和 SQL 示例，帮助系统更好地理解业务逻辑。

&emsp;&emsp;而核心代理层是整个系统的"大脑"，由 **NL2SQL Agent** 担任核心角色。Agent 负责理解用户意图、规划执行步骤、调度工具完成任务。它采用 **ReAct（推理-行动）模式**，即"思考一步、执行一步、再思考、再执行"的循环流程，直到完成任务。Agent 具备自主决策能力，能根据问题复杂度选择合适的工具组合，并在遇到错误时自动调整策略。这里我们参考 Vanna 项目的思路：https://github.com/vanna-ai/vanna/tree/main

<div align=center><img src="https://muyu20241105.oss-cn-beijing.aliyuncs.com/images/202511091520736.png" width=50%></div>

> 我们在源码级别上做的修改：
> 1. 重构了词嵌入函数，原生词嵌入模型效果不太理想，改用Jina的词嵌入模型，效果显著提升
> 2. 修改源码，使之支持中文sql生成中文question.
> 3. 重构函数，使之支持批量词嵌入调用与数据库批量插入。
> 4. 修改代码，相同问题将不在重复处理。


&emsp;&emsp;当用户提出问题后，数据经历如下流转：**用户输入** → **核心代理层（Agent 推理）** → **工具层（RAG 检索获取上下文）** → **基础设施层（Milvus 向量检索）** → **工具层（返回表结构、文档、历史 SQL）** → **核心代理层（生成 SQL）** → **工具层（验证语法、执行查询）** → **基础设施层（MySQL 执行）** → **核心代理层（格式化结果）** → **中间件层（实时推送）** → **用户交互层（展示结果）**。

&emsp;&emsp;RAG 技术是本系统的核心竞争力。当 Agent 需要生成 SQL 时，它首先调用 `get_table_schema` 工具，该工具将用户问题向量化后，在 Milvus 的三个集合中并行检索最相关的内容（表结构定义、业务说明、历史查询示例），将这些"上下文知识"拼接后返回给 Agent。Agent 基于这些上下文信息生成 SQL，准确率从无 RAG 时的约 40% 提升到 85% 以上。这种"检索外部知识增强生成"的方式，有效解决了 LLM 知识边界受限的问题。


&emsp;&emsp;而在技术选型上，我们使用的是**LangChain 1.0** 作为 Agent 框架，新版本提供了成熟的工具调度和中间件机制，大幅降低了开发复杂度。**Milvus** 作为向量数据库，具备开源、高性能、支持多种索引类型的优势，适合海量训练数据的语义检索。**Jina Embeddings v4** 提供 2048 维的高质量多模态向量，具有更好的中文支持和性价比。

# 二、前后端本地部署启动

&emsp;&emsp;我们首先介绍如何在本地服务器上快速部署和启动NL2SQL数据分析系统。其核心源码文件为：

<div align=center><img src="https://muyu20241105.oss-cn-beijing.aliyuncs.com/images/202511111432882.png" width=70%></div>

## 2.1 后端项目部署启动

- **Step 1. 安装 Anaconda/Miniconda**

&emsp;&emsp;如果系统中还没有安装 Conda，需要先安装，这里以 Anaconda3 为例：

```bash
    # 下载 Miniconda 安装脚本
    wget https://repo.anaconda.com/archive/Anaconda3-latest-Linux-x86_64.sh

    # 赋予执行权限
    chmod +x Anaconda3-latest-Linux-x86_64.sh

    # 运行安装脚本
    bash Anaconda3-latest-Linux-x86_64.sh
    # 按照提示完成安装，建议选择 yes 来初始化 conda
```

- **Step 2. 验证 Conda 安装**

```bash
    conda --version
```

&emsp;&emsp;正常应该看到类似 `conda 24.x.x` 的输出。

<div align=center><img src="https://muyu20241105.oss-cn-beijing.aliyuncs.com/images/202511101836902.png" width=70%></div>

- **Step 3. 创建 Conda 虚拟环境并激活**

&emsp;&emsp;为了避免依赖冲突，我们需要为NL2SQL数据分析系统的后端服务创建一个独立的虚拟环境：

```bash
    conda create -n nl2sqlagent python=3.13.5 -y
```

<div align=center><img src="https://muyu20241105.oss-cn-beijing.aliyuncs.com/images/202511102209961.png" width=70%></div>

&emsp;&emsp;然后激活虚拟环境：

```bash
    conda activate nl2sqlagent
```

&emsp;&emsp;激活成功后，命令行提示符会发生变化：

<div align=center><img src="https://muyu20241105.oss-cn-beijing.aliyuncs.com/images/202511102209962.png" width=70%></div>

- **Step 4. 安装后端项目依赖**

&emsp;&emsp;接下来一键安装后端依赖，执行如下命令：

```bash
    # 进入后端目录
    cd backend/vanna

    # 安装所有依赖
    pip install -r requirements.txt
```

<div align=center><img src="https://muyu20241105.oss-cn-beijing.aliyuncs.com/images/202511102209963.png" width=70%></div>

- **Step 5. 配置项目依赖**



&emsp;&emsp;然后配置环境变量。需要在 `backend/vanna/` 目录下创建 `.env` 文件，写入如下配置。注意：关于 Jina Embedding 和 Milvus Server的配置，请先按照下文的课件进行配置后，再进行对应的修改。

```env
    # OpenAI 兼容接口的 API Key
    # API_KEY=sk-proj-vFVUlYkytjG1REMdA8A
    # BASE_URL=https://ai.devtool.tech/proxy/v1
    # LLM_MODEL=gpt-5

    API_KEY=sk-62920a620fd70a0f
    BASE_URL=https://dashscope.aliyuncs.com/compatible-mode/v1
    LLM_MODEL=qwen-flash

    # LLM 生成参数（可选）
    LLM_TEMPERATURE=0.1
    LLM_MAX_TOKENS=14000

    # Agent 配置（可选）
    AGENT_RECURSION_LIMIT=50

    # ==================== Embedding 配置 ====================
    # 嵌入模型提供商: jina | qwen | bge (默认: jina)
    EMBEDDING_PROVIDER=jina
    # 嵌入模型 API 地址
    EMBEDDING_API_URL=http://192.168.110.131:8603/v1/embeddings
    # 嵌入模型 API Key（可选，Jina 本地部署不需要）
    # EMBEDDING_API_KEY=your-embedding-api-key
    # 嵌入模型名称（可选，不传则使用默认）
    # EMBEDDING_MODEL_NAME=jina-embeddings-v2-base-zh

    # ==================== Milvus 配置 ====================
    # Milvus 服务地址
    MILVUS_URI=http://192.168.110.131:19550
    # 向量相似度度量方式: COSINE | L2 | IP
    MILVUS_METRIC_TYPE=COSINE

    # ==================== MySQL 配置 ====================
    # MySQL 数据库主机
    MYSQL_HOST=192.168.110.131
    MYSQL_PORT=3306
    MYSQL_DATABASE=mydb
    MYSQL_USER=myuser
    MYSQL_PASSWORD=mypassword
```

&emsp;&emsp;接下来就可以启动后端项目服务了。执行如下代码：

```bash
    cd backend/vanna

    python api_server.py
```

<div align=center><img src="https://muyu20241105.oss-cn-beijing.aliyuncs.com/images/202511102209964.png" width=70%></div>


&emsp;&emsp;启动成功后，可以在在网页端通过`http://localhost:8000/docs`验证后端服务接口文档：


<div align=center><img src="https://muyu20241105.oss-cn-beijing.aliyuncs.com/images/202511111456003.png" width=70%></div>


## 2.2 前端项目部署启动

&esmp;&emsp;对于前端项目来说，同样是借助`React`框架搭建，所以在前端目录下执行：(注意：需要确定本地服务器上已经正确配置了 node.js 环境)

```bash
    cd frontend
    npm install
```

<div align=center><img src="https://muyu20241105.oss-cn-beijing.aliyuncs.com/images/202511102211497.png" width=70%></div>


```bash
    npm run dev
```

<div align=center><img src="https://muyu20241105.oss-cn-beijing.aliyuncs.com/images/202511102211498.png" width=70%></div>


&emsp;&emsp;稍等片刻会看到类似这样的输出：

```
    VITE v6.3.7  ready in 250 ms

    ➜  Local:   http://localhost:3000/
    ➜  Network: http://192.168.110.131:3000/
    ➜  press h + enter to show help
```

&emsp;&emsp;这表示开发服务器已成功启动！注意这几个关键信息：

- **Local**: `http://localhost:3000/` - 本地访问地址（只能在本机访问）
- **Network**: `http://192.168.110.131:3000/` - 网络访问地址（局域网内其他设备可访问）

&emsp;&emsp;在浏览器中打开 `http://localhost:3000`，就能看到系统的登录界面或首页。

<div align=center><img src="https://muyu20241105.oss-cn-beijing.aliyuncs.com/images/202511102211499.png" width=70%></div>

# 三、Jina Embedding 服务安装部署

&emsp;&emsp;Jina Embedding 在本项目中被设计为一个独立的微服务，通过 FastAPI 框架对外提供 HTTP API 接口。它的主要职责是将自然语言文本（用户问题、SQL 语句、表结构文档等）转换为高维稠密向量。举个例子：

```json
    问题：女性客户的平均消费金额是多少？
    系统操作：先在历史训练数据中找到语义最相似的案例（比如"不同性别客户的消费统计"），再生成准确的 SQL 查询。
```
&emsp;&emsp;这个"寻找相似案例"的过程，就依赖 Jina Embedding 将问题转化为向量，然后在向量数据库（Milvus）中进行相似度检索。同时，Jina Embedding 与 Milvus 结合应用。当系统需要添加训练数据（如表结构文档、SQL 案例）时，首先调用 Jina Embedding 将文本转为向量，然后将向量和原始文本一起存入 Milvus。后续用户提问时，系统会先将问题向量化，再从 Milvus 中检索出最相似的训练案例。


&emsp;&emsp;此外，我们在 Vanna 源码的基础上通过Jina Embedding 嵌入 NL2SQL 引擎协作工作流，大幅提升了 SQL 生成及信息检索的准确率。

&emsp;&emsp;接下来，我们就先详细介绍如何从零开始安装和部署 Jina Embedding 服务。

- **Step 1. 安装 Anaconda/Miniconda**

&emsp;&emsp;如果系统中还没有安装 Conda，需要先安装，这里以 Anaconda3 为例：

```bash
    # 下载 Miniconda 安装脚本
    wget https://repo.anaconda.com/archive/Anaconda3-latest-Linux-x86_64.sh

    # 赋予执行权限
    chmod +x Anaconda3-latest-Linux-x86_64.sh

    # 运行安装脚本
    bash Anaconda3-latest-Linux-x86_64.sh
    # 按照提示完成安装，建议选择 yes 来初始化 conda
```

- **Step 2. 验证 Conda 安装**

```bash
    conda --version
```

&emsp;&emsp;正常应该看到类似 `conda 24.x.x` 的输出。

<div align=center><img src="https://muyu20241105.oss-cn-beijing.aliyuncs.com/images/202511101836902.png" width=70%></div>

- **Step 3. 创建 Conda 虚拟环境并激活**

&emsp;&emsp;为了避免依赖冲突，我们为 Jina Embedding 服务创建一个独立的虚拟环境：

```bash
    conda create -n jina_run python=3.13.5 -y
```

<div align=center><img src="https://muyu20241105.oss-cn-beijing.aliyuncs.com/images/202511101836899.png" width=70%></div>

&emsp;&emsp;然后激活虚拟环境：

```bash
    conda activate jina_run
```

&emsp;&emsp;激活成功后，命令行提示符会发生变化：

<div align=center><img src="https://muyu20241105.oss-cn-beijing.aliyuncs.com/images/202511101836902.png" width=70%></div>

- **Step 4. 安装所需的 Python 包**

&emsp;&emsp;确保已经激活了 `jina_run` 环境后，执行如下命令安装依赖包：


```bash
    pip install fastapi==0.118.3 torch==2.8.0 transformers==4.56.2 Pillow==11.3.0 pydantic==2.10.3 uvicorn==0.38.0
    pip install --no-cache-dir torchvision --extra-index-url https://download.pytorch.org/whl/cu121
```

<div align=center><img src="https://muyu20241105.oss-cn-beijing.aliyuncs.com/images/202511101836901.png" width=70%></div>

- **Step 5. 准备 Jina Embedding 模型**

&emsp;&emsp;Jina Embedding 服务需要预训练的模型文件，这里大家直接在随课件存储的百度网盘中下载，并放在服务器上，如下所示：

<div align=center><img src="https://muyu20241105.oss-cn-beijing.aliyuncs.com/images/202511101848393.png" width=70%></div>

- **Step 6.  Jina Embedding API 启动**

&emsp;&emsp;同样，大家使用我们已经上传好的`app_jina_embedding_v4.py` 服务程序一键启动即可：

```bash
    CUDA_VISIBLE_DEVICES=2 \
    MODEL_DIR=/home/data/nongwa/workspace/model/embeddings/jina-embeddings-v4-vllm-retrieval \
    /root/anaconda3/envs/jina_run/bin/python -m uvicorn app_jina_embedding_v4:app \
    --host 192.168.110.131 --port 8898
```

- `CUDA_VISIBLE_DEVICES=0`: 指定使用第一块 GPU（编号从 0 开始）
  - 如果要使用第二块 GPU，改为 `CUDA_VISIBLE_DEVICES=1`
  - 如果要使用多块 GPU，改为 `CUDA_VISIBLE_DEVICES=0,1`
  - 如果不使用 GPU（仅用 CPU），去掉这个环境变量或设为 `-1`

- `MODEL_DIR=/home/MuyuWorkSpace/06_LangChainSqlAgent/models/jina-embeddings`:
  - 指定模型文件的目录路径
  - **请根据实际情况修改模型路径**

- `uvicorn app_jina_embedding_v4:app`: 启动 Uvicorn 服务器
  - `app_jina_embedding_v4`: Python 文件名（不含 .py 扩展名）
  - `app`: FastAPI 应用实例的变量名

- `--host 0.0.0.0`: 监听所有网络接口
  - 允许从其他机器访问此服务
  - 如果仅需本地访问，可改为 `--host 127.0.0.1`

- `--port 8000`: 指定服务端口为 8000
  - 可以根据需要修改为其他端口，如 `--port 8080`

<div align=center><img src="https://muyu20241105.oss-cn-beijing.aliyuncs.com/images/202511101907865.png" width=70%></div>

- **Step 7：接口请求测试**

In [5]:
import requests
import json

# API 端点
url = "http://192.168.110.131:8898/v1/embeddings"

# 请求数据
payload = {
    "inputs": [
        {"text": "人工智能正在改变世界"},
        {"text": "Machine learning is a subset of AI"}
    ],
    "normalize": True,
    "pooling": "mean"
}

# 发送 POST 请求
response = requests.post(url, json=payload)

# 打印结果
if response.status_code == 200:
    result = response.json()
    print(f"成功获取 {len(result['embeddings'])} 个向量")
    print(f"向量维度: {len(result['embeddings'][0])}")
    print(f"第一个向量的前10个值: {result['embeddings'][0][:10]}")
else:
    print(f"请求失败: {response.status_code}")
    print(response.text)

成功获取 2 个向量
向量维度: 2048
第一个向量的前10个值: [0.010142282582819462, -0.024486202746629715, 0.022074393928050995, 0.01172348391264677, 0.009627265855669975, -0.012862063013017178, -0.01761648990213871, -0.01902802102267742, -0.0048188273794949055, 0.019543739035725594]


&emsp;&emsp;同时后端能够正确的处理请求响应：

<div align=center><img src="https://muyu20241105.oss-cn-beijing.aliyuncs.com/images/202511101917490.png" width=70%></div>

- **Step 8：配置 .env 文件**

&emsp;&emsp; Jina Embedding API 启动并测试成功后，在 `.env` 文件配置相关参数，如下所示：

<div align=center><img src="https://muyu20241105.oss-cn-beijing.aliyuncs.com/images/202511101920473.png" width=70%></div>

# 四、Milvus 服务配置

&emsp;&emsp;这一小节，我们详细说明如何在本地环境中使用 Docker Compose 部署 Milvus 向量数据库，并通过 Python 代码进行功能测试。

&emsp;&emsp;Milvus 是一个云原生的向量数据库，专为处理大规模向量数据而设计。它支持多种索引算法（如 IVF、HNSW）和距离度量方式（如余弦相似度、欧氏距离），能够在毫秒级完成数十亿级向量的检索。


<style>
.center
{
  width: auto;
  display: table;
  margin-left: auto;
  margin-right: auto;
}
</style>

<p align="center"><font face="黑体" size=4>表：Milvus 核心概念</font></p>
<div class="center">

| 概念 | 说明 | 类比 |
|------|------|------|
| **Collection** | 向量集合，类似于数据库中的表 | MySQL 的 Table |
| **Field** | 字段，包括向量字段和标量字段 | MySQL 的 Column |
| **Entity** | 实体，一条完整的数据记录 | MySQL 的 Row |
| **Partition** | 分区，用于数据分片和查询优化 | MySQL 的 Partition |
| **Index** | 索引，加速向量检索 | MySQL 的 Index |
| **Metric Type** | 距离度量方式（L2/IP/COSINE） | 相似度计算方法 |

</div>

&emsp;&emsp;在本项目的 NL2SQL 系统中，Milvus 主要承担以下工作：

1. **训练数据存储**: 存储 SQL 语句、表结构文档等训练样本的向量表示；
2. **语义检索引擎**: 根据用户问题的向量，快速检索最相似的历史案例；
3. **知识库管理**: 支持训练数据的增删改查操作；

- **Step 1. 验证 Docker 环境**

&emsp;&emsp;首先在服务器上运行如下代码查看是否包含`Docker`环境:

```bash
    # 检查 Docker 版本
    docker --version
    # 输出示例: Docker version 24.0.6, build ed223bc

    # 检查 Docker Compose 版本
    docker compose version
    # 输出示例: Docker Compose version v2.23.0
```
<div align=center><img src="https://muyu20241105.oss-cn-beijing.aliyuncs.com/images/202511102131470.png" width=70%></div>

&emsp;&emsp;如未安装，可尝试如下方式进行快速配置：

```bash
    # 安装 Docker
    sudo yum install -y docker

    # 启动 Docker 服务
    sudo systemctl start docker
    sudo systemctl enable docker

    # 安装 Docker Compose
    sudo curl -L "https://github.com/docker/compose/releases/download/v2.23.0/docker-compose-$(uname -s)-$(uname -m)" -o /usr/local/bin/docker-compose
    sudo chmod +x /usr/local/bin/docker-compose

    # 验证安装
    docker --version
    docker-compose --version
```

- **Step 2. Milvus 一键启动**

&emsp;&emsp;接下来，大家进入到项目路径：milvus-deployment，直接通过 ./start.sh 一键启动

<div align=center><img src="https://muyu20241105.oss-cn-beijing.aliyuncs.com/images/202511102146179.png" width=70%></div>

- **Step 3. 验证测试**

&emsp;&emsp;同样，我们这里给大家提供了快速测试的脚本，为milvus-deployment/test_connection.py 文件，可以直接通过如下命令进行服务连通性测试：

```bash
   python test_connection.py 
```

- **Step 3：配置 .env 文件**

&emsp;&emsp;当测试通过后，在 `.env` 文件配置相关参数，如下所示：

<div align=center><img src="https://muyu20241105.oss-cn-beijing.aliyuncs.com/images/202511102149889.png" width=70%></div>

# 附录：Windows 安装 Node.js

&emsp;&emsp;`Node.js` 是一个基于 `Chrome V8` 引擎的 `JavaScript` 运行时环境，广泛应用于前端开发、后端服务器开发等场景。接下来我们详细介绍如何在 `Windows` 系统上从零开始安装和配置 `Node.js`，包括环境变量设置、`npm` 配置等关键步骤。

- **Step 1：下载 Node.js 安装包**

&emsp;&emsp;首先需要从官方网站下载适合你系统的 `Node.js` 安装包。`Node.js` 提供了稳定的 `LTS`（长期支持）版本和最新的 `Current` 版本，推荐选择 `LTS` 版本以确保稳定性。访问 `Node.js` 官网下载页：
   - 官方地址：[https://nodejs.org/en/download/](https://nodejs.org/en/download/)
   - 中文镜像：[https://nodejs.cn/download/](https://nodejs.cn/en/download)

&emsp;&emsp;优先选择 **LTS（长期支持）版本**，稳定性更好, 选择 `.msi` 安装包（根据系统架构选择 64-bit 或 32-bit）

<div align="center">
    <img src="https://muyu20241105.oss-cn-beijing.aliyuncs.com/images/202509181529316.png" alt="FuFanManus Architecture" width="800">
</div>

&emsp;&emsp;**点击下载**对应的 `.msi` 安装程序即可。

- **Step 2：运行安装程序**

&emsp;&emsp;下载完成后，需要运行 `.msi` 安装程序进行安装。安装过程中需要注意安装路径的选择以及环境变量的配置。首先找到下载的 `.msi` 文件，双击运行：

<div align="center">
    <img src="https://muyu20241105.oss-cn-beijing.aliyuncs.com/images/202509181535199.png" alt="FuFanManus Architecture" width="800">
</div>

&emsp;&emsp;安装向导出现后，点击 **Next / 下一步** 继续，阅读 `License Agreement`，勾选 "I accept the terms in the License Agreement"，点击 **Next**：

<div align="center">
    <img src="https://muyu20241105.oss-cn-beijing.aliyuncs.com/images/202509181535201.png" alt="FuFanManus Architecture" width="800">
</div>

&emsp;&emsp;这里强烈建议选择默认的路径：`C:\Program Files\nodejs\`，如必须修改，注意：路径中不要包含中文字符或特殊符号。

<div align="center">
    <img src="https://muyu20241105.oss-cn-beijing.aliyuncs.com/images/202509181535202.png" alt="FuFanManus Architecture" width="800">
</div>

&emsp;&emsp;自定义设置， 点击 "Install" 开始安装，系统可能会弹出权限确认，选择"是"或"允许"
   - 勾选 "Add to PATH" 选项
   - 勾选 "Automatically install npm package manager"
   - 这样可以在任何命令行窗口中直接使用 `node` 和 `npm` 命令

<div align="center">
    <img src="https://muyu20241105.oss-cn-beijing.aliyuncs.com/images/202509181535203.png" alt="FuFanManus Architecture" width="800">
</div>

&emsp;&emsp;等待安装完成即可。

# 附录：在 Ubuntu 系统上安装 Node.js

&emsp;&emsp;在实际的生产环境和服务器部署中，`Linux`系统（尤其是`Ubuntu`）是最常见的选择。相比`Windows`，`Linux`系统更加轻量、稳定，且对开发者友好。因此，掌握在`Ubuntu`上安装和配置`Node.js`是每一位全栈开发者的必备技能。

&emsp;&emsp;接下来，我将为大家详细讲解在`Ubuntu`系统上安装`Node.js`的方法。首先，我们需要更新系统的软件包列表，确保获取最新的软件信息：

```bash
    # 更新软件包索引
    sudo apt update

    # 升级已安装的软件包（可选）
    sudo apt upgrade -y
```

&emsp;&emsp;这个命令会联网下载最新的软件包列表，确保我们安装的是最新可用版本。接下来安装 Node.js 和 npm

```bash
    # 一条命令同时安装 Node.js 和 npm
    sudo apt install nodejs npm -y
```

&emsp;&emsp;等待安装完成后，验证安装：

```bash
    node -v    # 显示 Node.js 版本（可能是 v12 或 v14）
    npm -v     # 显示 npm 版本
```

<div align="center">
    <img src="https://muyu20241105.oss-cn-beijing.aliyuncs.com/images/202510231332604.png" alt="FuFanManus Architecture" width="800">
</div>